(where-mboostrs-stands-results-weaknesses-and-next-work)=
# Summary of benchmark results

The benchmark studies test what mboostrs can predict, discover, and report,
and how much time and analyst effort the workflow takes. Start with the
scorecard for the scope of the grades and the evidence behind them. The
[scoring rubric](overview.ipynb#scorecard-dimensions-at-a-glance) defines all
twelve dimensions and the 1–5 anchors.

(at-a-glance-strengths-and-gaps)=
## At a glance: where mboostrs does well and where it has work to do

Part V groups the benchmark pages into four tiers — **Headline competitive benchmarks**, **Discovery, causal, and agent workflows**, **Uncertainty and calibration studies**, and **Component ablations and engineering archive** — so you can read the bottom line first and drill down only where needed.

| Area | Status | What the benchmarks show | Primary study |
| --- | --- | --- | --- |
| **Planted distributional modeling** | **Strong** | Top five ranks overall across 21 arms on 16 planted distributions (`LSS_select_noncyclic` mean rank 5.25, `mb.fit` with `lss_trees=True` 5.34, default `mb.fit` 5.41, `LSS_select` 6.25, `quantiles` 6.44); 0.0–2.5% excess CRPS on additive parametric truths and 1.5–4.2% on the `x0·x1` interaction task. | [Distributional shootout](distributional_shootout.ipynb) |
| **Prediction-interval & probability calibration** | **Strong** | 90.2% median conformal coverage (**95.8%** within 0.85–0.95; S3 terciles 0.891 / 0.905 / 0.908), tied #1 binary ECE (**0.0221** with EBM), built-in `ConformalInterval.reliability` diagnostics, and best OpenML-CTR23 quantile calibration MAE (`mboostrs.quantiles` 0.024). | [Held-out comparison](heldout_competition.ipynb) · [Distributional shootout](distributional_shootout.ipynb) |
| **Real-data completion, discovery & accuracy** | **Strong** | Default `mb.fit` completes **285/285 (100.0%, zero failures)** real runs, runs in **1.52 s** median on the 50k × 20 session (4.4× faster than CatBoost) and **2.64 s** median across 320 runs (ahead of CatBoost 3.01 s), beats XGBoost and LightGBM on both regression and binary tracks, beats EBM on regression, matches CatBoost within statistical noise on both tracks (median ratio 1.030 regression, 1.015 binary), and recovers **80/80** planted predictors at **0.0% permutation FDR** (5/5 silent nulls). | [Held-out comparison](heldout_competition.ipynb) |
| **Fast real-data specification search (CTR23)** | **Strong** | `autospec(..., search="fast")` paired with the Rust `btree` histogram-subtraction engine completes 24/24 OpenML-CTR23 datasets (600/600 runs, 0 failures) in 1.43 s median (`autospec + pairs + residual tree`; 0.95 s additive `autospec`), reaching #2 mean R² (0.6387 across 20 workflows, within 0.002 of LightGBM's 0.6406 and ahead of CatBoost 0.4462, EBM 0.6308, MLP 0.6244, XGBoost 0.5870) and 0.7201 median R². | [24 real regression data sets](../cases/benchmark_realdata.ipynb) |
| **Interpretable additive, spatial & scaling fits** | **Strong** | 0.41 s additive fits at 90,000 rows; mixed numeric+categorical histogram-subtraction `btree`; $O(n+L)$ diagonal `brandom` for high-cardinality groups; quantile O-splines (`knot_placement="quantile"`); one-command `mb.reproduce`; audited decision trails and R `mboost`/`gamboostLSS` parity. | [Scaling](public_scaling.ipynb) · [Spatial](public_spatial.ipynb) |
| **Distributional unit invariance & non-cyclic tuning** | **Fixed (v2)** | Automatic `response_scale="auto"` eliminates the raw-unit failure (e.g., California housing CRPS 62,111 → 33,047); `tune_mstop` now tunes non-cyclic LSS directly. | [Distributional shootout](distributional_shootout.ipynb) · [LSS response scale](lss_response_scale.ipynb) |
| **High-order real-data distributional CRPS vs. GBDTs** | **Work to do** | On OpenML-CTR23 distributional CRPS, tree ensembles still lead on real data (LightGBM quantiles rank 3.48, QRF 5.16, CRPS forest 5.77, tuned hessboost 6.13 vs. `mboostrs.quantiles` 10.19, `mb.fit` 11.39, and `mboostrs.LSS_select_tree` 11.45). Adding joint trees (`LSS_select_tree`) cuts the gap over additive `LSS_select` by ~13% on real data, but near-deterministic physical surfaces still favour depth-6 GBDTs. | [Distributional shootout](distributional_shootout.ipynb) · [Tuned shootout](competitive_shootout.ipynb) |
| **Time-ordered forecast & causal effect intervals** | **Work to do** | Hourly PM2.5 bands covered 74% at nominal 80%; job-training individual-effect 90% intervals covered 61–66%. | [PM2.5 forecast](pm25_forecast_competition.ipynb) · [Causal discovery](causal_discovery.ipynb) |
| **Unmeasured scorecard dimensions** | **Work to do** | Open-structure Track A (D1a), human usability (D7), multi-analyst agent variance (D8), and general report faithfulness (D3) remain unmeasured. | [Scorecard rubric](overview.ipynb#scorecard-dimensions-at-a-glance) |

:::{tip} Rerunning and updating benchmarks
All maintained benchmarks are registered in `benches/registry.json` and driven by `benches/run.py`:
- `.venv/bin/python benches/run.py list` — list studies by tier and refresh mode.
- `.venv/bin/python benches/run.py status` — check artifact freshness against `HEAD`.
- `.venv/bin/python benches/run.py run <id> --workers 8 --update-book` — rerun a study, run its checkers, and re-execute its book page(s).
:::


(where-the-scorecard-stands)=
## mboostrs scorecard

The grade column of the table below is the recorded mboostrs profile under
`mboostrs-scorecard/2.1` in the **held-out comparison**. Six dimensions
received grades on the 1–5 scale. The comparison ran `mb.fit`, the automatic
workflow that searches for a model specification within a budget of fits and a
time limit and then fits it, and five other libraries at their defaults on 95
real tables and 35 simulations with known truth. The real tables had been used
in the development of `mb.fit`, which favours mboostrs. The last column lists evidence and gaps, including
findings from separately named studies. Each mark of **Unmeasured**
applies to this study's protocol. It makes no claim about the
library's ability. The [study](heldout_competition.ipynb) shows the
comparator grades, input conditions, and failures.

| Dimension | Held-out study grade | Evidence and remaining question |
| --- | --- | --- |
| [D1a](overview.ipynb#scorecard-d1a) Relationship recovery under a prescribed specification | Unmeasured | Needs a planted-truth test that supplies a rough formula and grades recovery only of structure the formula leaves open. |
| [D1b](overview.ipynb#scorecard-d1b) Relationship discovery from data alone | 5 / 5 | The common permutation rule found 80 of 80 planted predictors (including 10 of 10 interactions) at 0.0% FDR with 5 of 5 silent null tasks. The reported formula also achieved 80 of 80 (100%) TPR (including 10 of 10 interactions), 8.0% FDR, 5 of 5 silent null tasks, and 5 of 5 scale-only detections on S3; 108 of 114 (94.7%) real-data published effect directions matched. |
| [D2](overview.ipynb#scorecard-d2) Causal humility and support | Unmeasured | In the [job-training study](causal_discovery.ipynb), the mboostrs pipeline reported on that page chose a valid adjustment set on five of eight replicates; a separate mboostrs analyst run ([record](../appendix/records/agent_benchmarks.md)) chose one on all eight. Causal identification and sensitivity under other designs remain open. |
| [D3](overview.ipynb#scorecard-d3) Explanation quality | Unmeasured | The [hostile-workflow study](hostile_workflows_v4.ipynb) records correct verdicts on its planted traps. Faithfulness and completeness across a wider set of reports still need testing. |
| [D4](overview.ipynb#scorecard-d4) Importance quantification | Unmeasured | In the [job-training study](causal_discovery.ipynb), nominal 90% intervals for individual effects covered 66% of the true effects for the pipeline reported on that page and 61% in the separate analyst run, whose average-effect intervals covered 75%. |
| [D5](overview.ipynb#scorecard-d5) Experimental iteration speed | 3.5 / 5 | Median follow-up time was 1.52 seconds (p90 2.84 s, total 17.83 s) on the 50,000 × 20 session — 4.4× faster than CatBoost (6.69 s) and 45× faster than EBM (67.5 s). Median fit-plus-predict across 320 runs dropped to 2.64 s (ahead of CatBoost 3.01 s). |
| [D6](overview.ipynb#scorecard-d6) Robustness across conditions | 3 / 5 | 285 of 285 real-data runs completed (100.0%, zero failures, ahead of EBM at 260/285), leading all libraries on small-$n$ tasks (−9.5% excess) and S5 (0.281). |
| [D7](overview.ipynb#scorecard-d7) Human-friendliness | Unmeasured | Needs a human task study that measures time to a first useful discovery and recovery from errors. |
| [D8](overview.ipynb#scorecard-d8) Agent-friendliness | Unmeasured | The [hostile-workflow study](hostile_workflows_v4.ipynb) found correct core verdicts at different costs. One analyst per arm leaves variation across analysts unmeasured. |
| [D9](overview.ipynb#scorecard-d9) Uncertainty honesty (calibration) | 4.5 / 5 | Nominal 90% prediction intervals had 90.2% median coverage (95.8% within 0.85–0.95; S3 terciles 0.891 / 0.905 / 0.908), binary median ECE dropped to 0.0221 (tied best overall), and `ConformalInterval.reliability` warns on covariate shift and small calibration splits. The [time-ordered PM2.5 study](pm25_forecast_competition.ipynb) recorded 74% coverage for nominal 80% bands. |
| [D10](overview.ipynb#scorecard-d10) Reproducibility and auditability | 4 / 5 | Bit-identical prediction reproduction on 10/10 tasks, structured `DecisionTrail`, and one-command verification and report regeneration via `mb.reproduce` and `python -m mboostrs reproduce`. |
| [D11](overview.ipynb#scorecard-d11) Predictive validity | 4 / 5 | Completed 45/45 regression and 50/50 binary tasks; beats reference (37/45 regression, 37/50 binary), XGBoost (33/45, 37/50), LightGBM (28/45, 36/50), and EBM on regression (21/39), and matches CatBoost within statistical noise on both tracks (median ratios 1.030 and 1.015). Lowest loss on 6 of 7 planted tasks and 0.533 on S2. |

The study reports six grades, so the scorecard has no aggregate. Tuned
comparisons, historical rubrics, and separate workflow and causal studies
keep their own results below.


## What the studies establish

**Predictive accuracy (D11).** In the [held-out comparison](heldout_competition.ipynb),
mboostrs completes all 95 real tasks, beats XGBoost and LightGBM on both real
regression (median gap 8.0% vs. 9.9% and 19.5%) and binary classification
(6.0% vs. 30.4% and 35.7%), beats EBM on regression (21–18, median ratio 0.999),
and sits within statistical noise of CatBoost on both regression (median ratio
1.030, $p = 0.067$) and binary (median ratio 1.015, $p = 0.33$). The
[tuned shootout](competitive_shootout.ipynb) used ten real datasets and a
retrospective best-competitor comparison. Median excess loss in that study was
0.1% for CatBoost, 8.4% for mboostrs with a tree term, 38.3% for automatic
specification search, and 46.5% for the additive model. Several specialized
comparisons were closer. On the [Meuse zinc data](public_spatial.ipynb), the
held-out RMSE of log zinc was 0.300 for mboostrs with a tree on the coordinates
and 0.290 for CatBoost, where lower is better.

**Fitted versus reported discovery (D1b).** The
[held-out comparison](heldout_competition.ipynb) applied the same
held-out permutation rule (shuffle one column of the test rows and measure the
increase in test loss) to all fitted models on simulations with known effects.
For mboostrs, all 80 of 80 planted predictors (including 10 of 10 interactions)
were recovered with **0.0% FDR** and 5 of 5 silent null tasks, versus 31–46%
FDR for the tree libraries and EBM. The active-pruned formula returned to users
achieved **80 of 80 (100%) TPR** (including 10 of 10 interactions) with **8.0%** false discoveries, 5 of 5 silent null tasks, and 5 of 5
scale-only detections on S3. A separate
[historical planted-truth study](kdd_scorecard_study.ipynb) reported recovery of
main effects and a product term in its original workflow.

**Uncertainty (D4, D9).** Prediction-interval calibration concerns future
outcomes. Effect-interval calibration concerns unknown quantities such as
coefficients and treatment effects. The [held-out comparison](heldout_competition.ipynb)
found nominal 90% prediction intervals at 90.2% median coverage (95.8% within
0.85–0.95; S3 terciles 0.891 / 0.905 / 0.908) and binary median ECE of 0.0221
(tied best with EBM). The [time-ordered PM2.5 forecast](pm25_forecast_competition.ipynb)
achieved 74% coverage for nominal 80% bands. In the
[job-training study](causal_discovery.ipynb), nominal 90% intervals for
individual effects covered 66% of the true effects for the pipeline reported on
that page and 61% in a separate analyst run, whose average-effect intervals
covered 75%.

**Causal estimators (D2, D4).** The
[matched-nuisance comparison](causal_estimator_benchmark.ipynb) held the
nuisance learners (the models for the outcome and for the probability of
treatment, the propensity) and the confounders fixed for ten draws. Median
absolute error for the average treatment effect was $434 for `mb.aipw` at its
defaults, an augmented inverse-probability-weighted (AIPW) estimator, and $331
for EconML linear double machine learning. Both nominal 90% intervals covered
nine draws. AIPW divides by the estimated propensity. By default, `mb.aipw` clips it (bounds it to the range 0.01 to 0.99 so that the weights stay finite). Without that clipping, one mboostrs draw
had a $59,689 error. Ten draws give a small-sample comparison for
this design.

**Workflow cost (D5, D8).** The
[scaling study](public_scaling.ipynb) recorded a 0.41-second additive fit
at 90,000 rows. The [held-out comparison](heldout_competition.ipynb)
measured a different operation: automatic selection plus prediction and
permutation importance, with a 1.52-second median follow-up at 50,000 rows
(4.4× faster than CatBoost's 6.69 s and 45× faster than EBM's 67.5 s) and
2.64-second median fit-plus-predict across the 320-run battery (ahead of
CatBoost's 3.01 s).
In the [hostile-workflow study](hostile_workflows_v4.ipynb), both mboostrs
analysts reached the core verdicts. The analyst given the mboostrs agent
skill (the library's bundled guide for AI agents) used 127 minutes and 686
fits. The analyst without it used 87 minutes and 378 fits. The study
used one analyst per arm.

**Completion and auditability (D6, D10).** The
[held-out comparison](heldout_competition.ipynb) completed 285 of 285
mboostrs runs (100.0%, zero failures, ahead of EBM at 260/285). A separate
[predictive refresh](competitive_refresh.ipynb) completed 37 of 45 jobs at its
small search budget. Repeated predictions match bit-identically (10/10), and
`mb.reproduce` plus `python -m mboostrs reproduce` verify a saved decision
trail and regenerate its report in one command.


## Where each result is reported

The linked chapters give the data, split, comparators, and measurement units
for their results. The [benchmark records appendix](../appendix/benchmark_records.md)
includes the original protocols and run reports. Use those records to check
denominators, versions, deviations, and the evidence behind a grade.

| Question | Studies to read |
| --- | --- |
| How does automatic `mb.fit` compare with its previous version and with other libraries at defaults? | [Held-out comparison](heldout_competition.ipynb) |
| How does fast `autospec(..., search="fast")` perform across real tabular regression datasets? | [24 real regression data sets](../cases/benchmark_realdata.ipynb) |
| How do the results change with tuning or an explicit model structure? | [Tuned shootout](competitive_shootout.ipynb), [predictive refresh](competitive_refresh.ipynb), [capacity chooser](chooser_shootout.ipynb), [interaction capacity](interaction_capacity.ipynb) |
| How do predictive comparisons vary by outcome, spatial design, or table size? | [Counts](public_count.ipynb), [binary outcomes](public_binary.ipynb), [spatial data](public_spatial.ipynb), [distributional prediction](public_distributional.ipynb), [distributional shootout](distributional_shootout.ipynb), [scaling](public_scaling.ipynb) |
| What drives a prediction or interval failure? | [Stopping rule](stopping_rule_ablation.ipynb), [joint regularization](joint_regularization_pilot.ipynb), [estimation and discovery](paired_estimation_discovery.ipynb), [interval calibration](interval_calibration.ipynb) |
| What can analysts discover with the library, and how do causal estimators perform? | [Claim-cost discovery](claims_discovery.ipynb), [job-training analysis](causal_discovery.ipynb), [hostile workflows](hostile_workflows_v4.ipynb), [causal estimators](causal_estimator_benchmark.ipynb), [historical planted-truth study](kdd_scorecard_study.ipynb) |

(what-is-being-fixed-next)=
## Open problems and the evidence needed

| Observed shortfall | A follow-up that would establish progress |
| --- | --- |
| **Development exposure on the 95 real tables (D11).** Default `mb.fit` now completes 285/285 runs and matches CatBoost within statistical noise on both tracks, but those 95 real tables informed remediation. [Held-out study](heldout_competition.ipynb) | Confirm automated predictive accuracy on an independent held-back task suite outside the 95 development tables. |
| **Sub-second 50k sessions and 5M-row stress scaling (D5, D6).** Median 50k × 20 follow-up latency dropped to 1.52 s (ahead of CatBoost and EBM), behind sub-second XGBoost (0.58 s) and LightGBM (0.76 s). [Held-out study](heldout_competition.ipynb) | Bring 50k × 20 automated follow-ups under 1.0 s and add a frozen 5M-row stress benchmark. |
| **Effect and forecast undercoverage (D4, D9).** Job-training effect intervals were too narrow, and hourly forecast bands fell below nominal coverage. [Job-training study](causal_discovery.ipynb) · [PM2.5 forecast](pm25_forecast_competition.ipynb) | Measure effect coverage on fresh simulated draws with known effects and weak overlap. Measure shift-warning recall and forecast coverage on a fresh time-ordered test period. |
| **Agent workflow fit counts (D8).** Earlier guided-agent runs consumed more fits (~686 vs. ~378) than unguided runs before fast search became the default. [Hostile workflows](hostile_workflows_v4.ipynb) | Measure multi-analyst agent variance and fit counts with the updated fast search and agent skill under a fresh frozen protocol. |
| **Evidence gaps (D1a, D3, D7).** Current results leave open-structure recovery, general report quality, and human usability unmeasured. | Give analysts a rough formula with held-back effects to find (Track A in the [information contracts](overview.ipynb#information-contracts-511)) and score only those effects. Run a human task study and score report faithfulness. |


(what-this-page-does-not-establish)=
## How far the results extend

- Each grade and metric applies to its recorded input conditions, split,
  version, and workflow. Comparisons of individual fits, automatic model
  search, and agent sessions use different time boundaries.
- A held-out split protects the test rows within a task. Generalization to
  new task families requires independent tasks. Several studies reused
  datasets during development.
- Historical grades describe their original rubrics. A change to the
  library changes a recorded grade only when a new experiment measures it.